# Implementación de la partición de caudal

**Dataset:** RB-1521 · **Cuaderno:** 04 · **Tema:** función determinística y día de referencia

## Introducción

La identidad del cuaderno anterior se escribe una vez, en `rb1521_particion.particionar`, y el motor la llama. El día de referencia es la primera fila, 2020-05-02, porque es el inicio de la serie continua y en esa fila la partición cierra. No es un día elegido por ser el más favorable entre varios candidatos: es el origen temporal del archivo.

## Objetivos

Reproducir Q-water y Q-oil del 2020-05-02, y mostrar que un caudal distinto del almacenado produce otro reparto sin escribir el Excel.

## Metodología

El motor lee la copia de trabajo, comprueba el hash y aplica la partición a las entradas recibidas. Si no llegan entradas, usa Q-fluidos y BSW de la primera fila. `recomendacion_operacional_autorizada` permanece en falso: el reparto no es una consigna de maniobra.

## Ecuación

\[
Q_{agua} = Q_{fluidos} \times \mathrm{BSW}, \qquad Q_{petroleo} = Q_{fluidos} - Q_{agua}
\]

\(Q_{fluidos}\) es el valor de la columna o el valor sustituido en el escenario. BSW es la fracción de la columna o la sustituida. El resultado se compara con la fila solo cuando las entradas son las de esa fila.


In [1]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from evaluacion.motor import evaluar_solicitud
base = evaluar_solicitud({"dataset_id": "RB-1521", "pozo_id": "RB-1521", "variables": []})
esc = evaluar_solicitud({
    "dataset_id": "RB-1521",
    "pozo_id": "RB-1521",
    "variables": [{"nombre": "caudal_fluidos", "valor": 8500, "unidad": "BFPD"}],
})
print("referencia", base["fecha_referencia"], base["modelos"][0]["valores"])
print("coincide con la fila", base["coincidencia_dia_referencia"])
print("escenario 8500", esc["modelos"][0]["valores"]["q_water"], esc["modelos"][0]["valores"]["q_oil"])
print("el escenario ya no es la fila", esc["coincidencia_dia_referencia"])


referencia 2020-05-02 {'q_fluidos': 8977.0, 'bsw': 0.985, 'q_water': 8842.345, 'q_oil': 134.65500000000065, 'dias_serie': 129.0, 'dias_identidad_exacta': 128.0}
coincide con la fila True
escenario 8500 8372.5 127.5
el escenario ya no es la fila False


## Resultados

El 2020-05-02 el motor devuelve Q-water = 8842,345 y Q-oil = 134,655, que son las cifras almacenadas. Con Q-fluidos = 8500 y el BSW de referencia 0,985, Q-water pasa a 8372,5. La copia del Excel no cambia: la función no tiene una ruta de escritura.

El motor rechaza un pozo de BES-jun y rechaza la frecuencia como entrada, porque la frecuencia no interviene en la partición.

## Interpretación

El simulador puede mover Q-fluidos y BSW dentro de los extremos observados en las 129 filas. Esos extremos describen la serie; no son un límite de bomba. Cambiar la frecuencia no tiene un efecto calculable con las fórmulas que este archivo permite verificar.

## Conclusión

La implementación queda limitada a la partición, con trazabilidad al día de referencia y sin escritura del original.
